# RQ3 – Tổng quát hóa chéo nguồn dữ liệu

Notebook này **chỉ đọc artifact** của các run đã hoàn tất và gọi hàm trong
`scripts/analyze_rq3.py`; không huấn luyện lại và không đánh giá lại test.

Ba protocol (plan: `plans/rq3/cross_dataset_generalization.yaml`):

- **P1:** train RealWaste, test toàn bộ TrashNet.
- **P2:** train TrashNet, test toàn bộ RealWaste.
- **P3:** train pooled (cả hai nguồn), test phần held-out của từng nguồn.

Tiêu chí đăng ký trước: pooled cao hơn single-source ≥ 0.02 Macro-F1 trên **cả hai** nguồn.

> Cần đặt `WASTE_DATA_ROOT` / `WASTE_EXPERIMENTS_ROOT` nếu dữ liệu và kết quả nằm ngoài repo
> (xem `docs/gpu_local.md`).

## 0. Chuẩn bị

Thêm repo root và thư mục `scripts/` vào `sys.path`, sau đó in ra đường dẫn đang dùng
để tránh đọc nhầm thư mục kết quả.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# PDF qua LaTeX không dựng được bảng HTML của pandas; hiển thị dạng text để xuất PDF ổn định.
pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

import analyze_rq3  # noqa: E402
from src.utils.paths import data_root, experiments_root  # noqa: E402

print("repo:", ROOT)
print("data_root:", data_root())
print("experiments_root:", experiments_root())

## 1. Kiểm tra artifact

Sinh lại bảng/hình vào `docs/RQ3/` từ các run `completed`, rồi xem nhanh số run và seed
của từng protocol.

In [ ]:
analyze_rq3.main(
    [
        "--output-md",
        str(ROOT / "docs/RQ3/rq3_results.md"),
        "--output-csv",
        str(ROOT / "docs/RQ3/rq3_metrics.csv"),
        "--figures-dir",
        str(ROOT / "docs/RQ3/rq3_analysis"),
    ]
)

data = analyze_rq3.load_runs(experiments_root())
pd.DataFrame(
    [
        {
            "experiment": experiment,
            "n_runs_completed": len(runs),
            "seeds": sorted(run["run"].get("seed") for run in runs),
        }
        for experiment, runs in data.items()
    ]
)

## 2. Bảng tổng hợp

Mean ± std trên 3 seed. `Gap val−test` = Macro-F1 validation in-domain trừ Macro-F1 test;
gap lớn nghĩa là mô hình rơi mạnh khi rời miền huấn luyện.

In [ ]:
rows = analyze_rq3._run_rows(
    data, analyze_rq3.heldout_test_paths(ROOT / "data/splits/v1/split.csv")
)
agg = analyze_rq3.aggregate(rows)
agg[
    [
        "label",
        "train",
        "test_source",
        "n_seeds",
        "n_test",
        "macro_f1_mean",
        "macro_f1_std",
        "accuracy_mean",
        "best_val_macro_f1",
    ]
].round(4)

## 3. Khoảng cách domain

So sánh trực tiếp validation in-domain và test: P1/P2 lệch miền hoàn toàn, P3 test
trên phần held-out của chính hai nguồn đã tham gia train.

In [ ]:
analyze_rq3.domain_gap(agg).round(4)

## 4. Pooled so với single-source

**Lưu ý phương pháp:** single-source test trên toàn bộ nguồn còn lại (2.390 / 3.092 ảnh),
pooled test trên held-out của split v1 (360 / 463 ảnh). Vì vậy bảng dưới có cả cột
`*_heldout`: chấm lại **cùng held-out test** bằng cách lọc predictions đã có (không
đánh giá lại test).

**Mix-ratio:** realwaste giữ 100% train, tăng dần tỉ lệ group TrashNet (0 / 25 / 50 / 75%),
mức 100% là pooled hiện có.


In [ ]:
comparison = analyze_rq3.compare_pooled(agg)
comparison.round(4)

In [ ]:
analyze_rq3.paired_delta(rows).round(4)


## 5. Cần bao nhiêu dữ liệu nguồn thứ hai?

Macro-F1 trên held-out test của từng nguồn theo tỉ lệ TrashNet trong train.
Tiêu chí đã đăng ký trong `plans/rq3/source_mix.yaml`.


In [ ]:
dose = analyze_rq3.dose_response(agg)
print(analyze_rq3.dose_verdict(dose))
dose.round(4)

## 6. Per-class F1 trên test

Lớp nào mất nhiều nhất khi chuyển miền; dùng để giải thích cơ chế domain shift
(nền ảnh, ánh sáng, cách chụp).

In [ ]:
per_class = analyze_rq3.per_class_table(rows)
per_class.pivot_table(index=["label", "test_source"], columns="class", values="f1_mean").round(3)

## 7. Sụp đổ dự đoán

Tỉ lệ nhãn dự đoán nhiều nhất trên test. Tỉ lệ cao (đặc biệt khi test là toàn bộ nguồn
còn lại) cho thấy mô hình dồn về một lớp thay vì phân biệt nội dung.

In [ ]:
shift = analyze_rq3.prediction_shift(data)
shift[["label", "test_source", "top_predicted", "top_share"]].round(3)

## 8. Hình

Ba hình do `scripts/analyze_rq3.py` sinh từ artifact (không đánh giá lại test).

In [ ]:
from IPython.display import Image, display  # noqa: E402

figures = ROOT / "docs/RQ3/rq3_analysis"
for name in ("test_macro_f1.png", "dose_response.png", "confusion_matrices.png"):
    display(Image(filename=str(figures / name)))

## 9. Kết luận ngắn

- P1/P2 cho thấy domain shift rất mạnh: Macro-F1 test ngoài miền thấp hơn nhiều so với
  validation in-domain, và dự đoán dồn về một lớp.
- P3 giữ Macro-F1 cao trên held-out của cả hai nguồn, tức là khi mô hình đã thấy dữ liệu
  của cả hai miền thì không còn phải "chuyển miền" nữa.
- Tiêu chí đăng ký trước đạt trên cả hai nguồn, nhưng phải đọc kèm lưu ý ở mục 4.
- Mix-ratio cho biết cần bao nhiêu dữ liệu nguồn thứ hai để đạt mức pooled; xem mục 5.
- Số liệu chi tiết: `docs/RQ3/rq3_results.md`, `docs/RQ3/rq3_metrics.csv`.
- Notebook commit ở trạng thái sạch output, kèm PDF trong `docs/notebooks/rq3/`.